# T-Tests
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** why the t-distribution is used instead of the normal distribution when σ is unknown
- **Calculate** a one-sample and two-sample t-test statistic
- **Interpret** the degrees of freedom and how they affect the shape of the t-distribution

> **Tip:** Start by moving the **degrees of freedom slider** from 1 to 30, watch the t-distribution's heavy tails gradually converge to the standard normal curve.

---
## How we got here

In *17–19* we built hypothesis tests using the standard normal distribution and known population parameters. In real life, we almost never know the population standard deviation σ, we have to estimate it from our sample. The t-test is the solution: it uses the sample standard deviation s instead of σ, but adjusts for the extra uncertainty that introduces by using the t-distribution instead of the normal.

---
## Why this matters for data science

The two-sample t-test is arguably the most commonly run statistical test in industry. Every time a product team compares user engagement between a control and treatment group in an A/B test with moderate sample sizes, they run a t-test (or its equivalent). Understanding degrees of freedom and when the t-distribution matters versus when you can approximate it with the normal distribution is a practical skill you will use constantly.

---
## Try it yourself

In [2]:
from tkh_utils import make_slider, make_int_slider, make_dropdown, make_output, display_widget, register_observer

_x_range = np.linspace(-5, 5, 500)

out = make_output()
test_dropdown = make_dropdown(
    ["One-sample", "Two-sample (independent)", "Paired"],
    "One-sample", "Test type:",
)
n1_slider    = make_int_slider(5, 100, 1, 20, "n (or n₁):")
xbar1_slider = make_slider(0.0, 5.0, 0.1, 2.0, "x̄ (or x̄₁):")
s1_slider    = make_slider(0.5, 4.0, 0.1, 1.5, "s (or s₁):")
n2_slider    = make_int_slider(5, 100, 1, 20, "n₂:")
xbar2_slider = make_slider(0.0, 5.0, 0.1, 0.5, "x̄₂:")
s2_slider    = make_slider(0.5, 4.0, 0.1, 1.5, "s₂:")

def render(test_type, n1, xbar1, s1, n2, xbar2, s2):
    if test_type == "One-sample":
        t_stat = (xbar1 - 0.0) / (s1 / np.sqrt(n1))
        df = n1 - 1
    elif test_type == "Two-sample (independent)":
        se = np.sqrt(s1**2 / n1 + s2**2 / n2)
        t_stat = (xbar1 - xbar2) / se if se > 0 else 0
        num = (s1**2 / n1 + s2**2 / n2)**2
        den = (s1**2 / n1)**2 / max(n1 - 1, 1) + (s2**2 / n2)**2 / max(n2 - 1, 1)
        df = num / den if den > 0 else n1 + n2 - 2
    else:  # Paired
        t_stat = xbar1 / (s1 / np.sqrt(n1))
        df = n1 - 1

    df = max(df, 1)
    p_val  = 2 * stats.t.sf(abs(t_stat), df)
    t_crit = stats.t.ppf(0.975, df)

    y_t      = stats.t.pdf(_x_range, df)
    x_right  = _x_range[_x_range >=  t_crit]
    x_left   = _x_range[_x_range <= -t_crit]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=_x_range, y=stats.norm.pdf(_x_range),
        mode="lines", line=dict(color=PALETTE["muted"], width=1.5, dash="dot"),
        name="Normal (reference)",
    ))
    fig.add_trace(go.Scatter(
        x=_x_range, y=y_t,
        mode="lines", line=dict(color=PALETTE["primary"], width=3),
        name=f"t-distribution (df = {df:.0f})",
    ))
    for xs in [x_right, x_left]:
        if len(xs) > 1:
            fig.add_trace(go.Scatter(
                x=np.concatenate([[xs[0]], xs, [xs[-1]]]),
                y=np.concatenate([[0], stats.t.pdf(xs, df), [0]]),
                fill="toself", fillcolor="rgba(247,103,7,0.30)",
                line=dict(width=0), showlegend=False,
            ))
    t_plot = float(np.clip(t_stat, -5, 5))
    fig.add_vline(
        x=t_plot,
        line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
        annotation_text=f"t = {t_stat:.2f}",
    )
    layout = base_layout(
        title=f"t = {t_stat:.2f}, df = {df:.0f}, p = {p_val:.4f}",
        xaxis_title="Test Statistic (t)",
        yaxis_title="Probability Density",
    )
    layout.update(xaxis=dict(range=[-5, 5]))
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def _on_change(change):
    render(
        test_dropdown.value,
        n1_slider.value, xbar1_slider.value, s1_slider.value,
        n2_slider.value, xbar2_slider.value, s2_slider.value,
    )

register_observer(
    [test_dropdown, n1_slider, xbar1_slider, s1_slider, n2_slider, xbar2_slider, s2_slider],
    _on_change,
)
display_widget(
    [test_dropdown, n1_slider, xbar1_slider, s1_slider, n2_slider, xbar2_slider, s2_slider],
    out,
)
render(
    test_dropdown.value,
    n1_slider.value, xbar1_slider.value, s1_slider.value,
    n2_slider.value, xbar2_slider.value, s2_slider.value,
)


---
## What's happening?

When σ is unknown and must be estimated by the sample standard deviation s, the test statistic follows a **t-distribution** with n−1 degrees of freedom, not a standard normal.

| Test type | Formula | Degrees of freedom |
|-----------|---------|-------------------|
| One-sample | t = (x̄ − μ₀) / (s/√n) | df = n − 1 |
| Independent two-sample | t = (x̄₁ − x̄₂) / SE_pooled | df ≈ n₁ + n₂ − 2 |
| Paired | t = d̄ / (s_d/√n) | df = n − 1 |

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

### Why the t-distribution has heavier tails

With a small sample, the estimate s is itself uncertain, sometimes much too small, sometimes much too large. That extra uncertainty spreads the distribution into heavier tails, requiring a more extreme test statistic to reach the same significance level as the normal distribution.

As df → ∞ (i.e., as n grows), the t-distribution converges to the standard normal, confirming that when n is large, s ≈ σ and the distinction disappears.

---
## Real-world example: Comparing two drug treatments

A clinical researcher wants to compare the mean reduction in systolic blood pressure (mmHg) for two drugs, given to independent groups of 20 patients each. This is the prototypical independent two-sample t-test scenario.

The chart shows the simulated distributions of BP reductions for both drugs and marks the observed group means. Notice:

- **Notice:** There is natural variability within each group, individual responses vary even under the same treatment, which is why we need a test rather than just reading off the means
- **Notice:** The two means are different, but given the within-group spread, the question is whether that difference is too large to be chance
- **Notice:** The t-test accounts for both the size of the difference and the variability within groups, a larger within-group spread requires a larger mean difference to reach significance

> **Discussion question:** Drug A reduces BP by a mean of 8 mmHg (s = 5) and Drug B by 11 mmHg (s = 12) in samples of n = 20. Without running the test, which drug do you think will show a more significant result, and why does the standard deviation matter?

In [3]:
np.random.seed(404)

# ── Two-sample t-test: BP reduction by two drugs ─────────────────────────────────
# Parameters calibrated to typical antihypertensive clinical trial ranges
n_a, mu_a, sd_a = 20, 8.2,  5.1   # Drug A: modest effect, consistent
n_b, mu_b, sd_b = 20, 11.5, 11.8  # Drug B: larger effect, high variability

drug_a = np.random.normal(mu_a, sd_a, n_a)
drug_b = np.random.normal(mu_b, sd_b, n_b)

t_stat, p_val = stats.ttest_ind(drug_b, drug_a)
df = n_a + n_b - 2

fig = go.Figure()
for data, name, color, mu in [
    (drug_a, f"Drug A  x̄={np.mean(drug_a):.1f}", PALETTE["primary"],   mu_a),
    (drug_b, f"Drug B  x̄={np.mean(drug_b):.1f}", PALETTE["secondary"], mu_b),
]:
    fig.add_trace(go.Box(
        y=data, name=name,
        marker_color=color, boxmean=True,
        boxpoints="all", jitter=0.3, pointpos=0,
        marker=dict(size=5, opacity=0.5),
    ))

layout = base_layout(
    title=f"BP Reduction by Drug — Independent Two-Sample T-Test<br>"
          f"t({df}) = {t_stat:.2f},  p = {p_val:.3f}",
    xaxis_title="Drug",
    yaxis_title="Systolic BP Reduction (mmHg)",
)
fig.update_layout(layout)
fig.show()

### Choosing the right t-test

| Scenario | Test | Key assumption |
|----------|------|---------------|
| Compare sample mean to known value | One-sample t-test | Data is approximately normal |
| Compare means of two independent groups | Independent two-sample t-test | Groups are independent; roughly equal variance |
| Compare same subjects under two conditions | Paired t-test | Each pair is matched (before/after, twins, etc.) |
| Compare means of 3+ groups | ANOVA (not a t-test) | Use F-distribution; t-test doesn't extend directly |
| Very large samples (n > 200) | Z-test or t-test (equivalent) | CLT ensures normality; t ≈ z at large df |

---
## Extending to 3+ groups: ANOVA

The t-test compares exactly two group means. When you have three or more groups, running multiple t-tests inflates the false-positive rate (see *18: P-Values — multiple testing*). **ANOVA (Analysis of Variance)** solves this by testing all groups simultaneously with a single F-statistic.

The F-statistic asks: is the variance *between* groups larger than what you would expect from random sampling variation *within* groups?

$$F = \frac{\text{Variance between groups}}{\text{Variance within groups}} = \frac{MS_{\text{between}}}{MS_{\text{within}}}$$

A large F means the groups are more different from each other than chance would predict — evidence that at least one group mean is different from the others.

| Test | Groups | Statistic | Null hypothesis |
|------|--------|-----------|----------------|
| t-test | 2 | t | μ₁ = μ₂ |
| One-way ANOVA | 3+ | F | μ₁ = μ₂ = μ₃ = … |

> **Important:** A significant ANOVA F-test only tells you that *at least one* group differs from the others. It does not tell you *which* groups differ. Post-hoc tests (Tukey HSD, Bonferroni) are needed to identify the specific pairs.

In [4]:
np.random.seed(77)

# Compare accuracy scores of three model configurations
config_a = np.random.normal(0.81, 0.04, 30)  # baseline
config_b = np.random.normal(0.83, 0.04, 30)  # feature engineering added
config_c = np.random.normal(0.88, 0.04, 30)  # hyperparameter tuned

f_stat, p_anova = stats.f_oneway(config_a, config_b, config_c)

fig = go.Figure()
for data, label, color in [
    (config_a, f"Config A  x̄={np.mean(config_a):.3f}", PALETTE["primary"]),
    (config_b, f"Config B  x̄={np.mean(config_b):.3f}", PALETTE["secondary"]),
    (config_c, f"Config C  x̄={np.mean(config_c):.3f}", PALETTE["accent"]),
]:
    fig.add_trace(go.Box(
        y=data * 100, name=label,
        marker_color=color, boxmean=True,
        boxpoints="all", jitter=0.3, pointpos=0,
        marker=dict(size=5, opacity=0.50),
    ))

sig = "significant" if p_anova < 0.05 else "not significant"
layout = base_layout(
    title=(
        f"One-Way ANOVA: Model Accuracy Across 3 Configurations\n"
        f"F = {f_stat:.2f},  p = {p_anova:.4f}  ({sig} at α = 0.05)"
    ),
    xaxis_title="Model Configuration",
    yaxis_title="Accuracy (%)",
)
fig.update_layout(layout)
fig.show()

---
## Key takeaway

> **The t-test replaces the z-test when σ is unknown, using heavier-tailed t-distributions to account for that uncertainty — as sample size grows, the t-distribution converges to normal and the distinction disappears.**

---
*Next up: Chi-Square Test — when you need to test relationships between categorical variables rather than means*